![banner](https://github.com/hello-robot/stretch_mujoco/raw/main/docs/images/stretch_mujoco.png)

<h1><center>Getting Started Tutorial  <a href="https://colab.research.google.com/github/hello-robot/stretch_mujoco/blob/main/docs/getting_started.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook provides an introduction to Stretch's [**MuJoCo** simulation](https://github.com/hello-robot/stretch_mujoco/). You can run this notebook with either **CPU** or **GPU** instance.


## Install


In [30]:
# # Set up Stretch Mujoco repo
# # %rm -rf ./stretch_mujoco/
# # # !git clone https://github.com/hello-robot/stretch_mujoco --recurse-submodules
# # !git clone https://github.com/hello-robot/stretch_mujoco --depth 1
# # %cd ./stretch_mujoco/
# # %pip install -e ".[jupyter]"

# # Check if we can use GPU rendering
# import os
# import subprocess
# try:
#     subprocess.run('nvidia-smi')
#     USE_GPU=True
# except:
#     USE_GPU=False

# # Setup rendering
# if USE_GPU:
#     # Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
#     # This is usually installed as part of an Nvidia driver package, but the Colab
#     # kernel doesn't install its driver via APT, and as a result the ICD is missing.
#     # (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
#     NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
#     if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
#         with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
#             f.write("""{
#       "file_format_version" : "1.0.0",
#       "ICD" : {
#           "library_path" : "libEGL_nvidia.so.0"
#       }
#   }""")

#     # Configure MuJoCo to use the EGL rendering backend (requires GPU)
#     print('Setting environment variable to use GPU rendering:')
#     %env MUJOCO_GL=egl
# else:
#     # Required for OSMesa OpenGL driver
#     !apt-get update
#     !apt-get install -y libosmesa6-dev libgl1-mesa-glx libglfw3

#     print('Setting environment variable to use CPU rendering:')
#     %env MUJOCO_GL=osmesa

# # Other imports and helper functions
# import time
# import pprint
# import itertools
# import numpy as np

# # Graphics and plotting.
# !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
# !pip install -q mediapy
# import mediapy as media
# import matplotlib.pyplot as plt

# # More legible printing from numpy.
# np.set_printoptions(precision=3, suppress=True, linewidth=100)

After installing, we need robocasa set up


In [31]:
# python -m pip install -e ".[robocasa]"
# python -m pip install -e third_party/robosuite
# python -m pip install -e third_party/robocasa

If you get an error like

(stretch_mujoco_v310) orrijoa@orrijoa-IdeaPad-Gaming-3-15ACH6:~/projects/stretch_mujoco_jupyter/stretch_mujoco$ python -m pip install -e third_party/robosuite
Obtaining file:///home/orrijoa/projects/stretch_mujoco_jupyter/stretch_mujoco/third_party/robosuite
ERROR: file:///home/orrijoa/projects/stretch_mujoco_jupyter/stretch_mujoco/third_party/robosuite does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

From the root of your stretch_mujoco repo, run:
git submodule update --init


In [32]:
# python third_party/robosuite/robosuite/scripts/setup_macros.py
# python third_party/robocasa/robocasa/scripts/setup_macros.py
# python third_party/robocasa/robocasa/scripts/download_kitchen_assets.py

## Basics


The `StretchMujocoSimulator` class is used to:

- Start/stop the simulation
- Read camera imagery
- Read lidar scans
- Read joint states
- Position control the robot's ranged joints
- Velocity control the robot's mobile base


### Prerequisites to run to stream cameras in each windows


In [33]:
# Viewer on NVIDIA dGPU (windowed)
import os
os.environ.pop("MUJOCO_GL", None)              # ensure not headless
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"

# If you want headless instead, comment the above and use:
# import os, shutil
# os.environ["MUJOCO_GL"] = "egl" if shutil.which("nvidia-smi") else "osmesa"

In [34]:
import time
import threading
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt
import cv2

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

# Optional dependency
try:
    import mediapy as media
    HAS_MEDIAPY = True
except ImportError:
    HAS_MEDIAPY = False
    print("mediapy not installed. Install with: python -m pip install mediapy")
    
# Project imports last (after env is set!)
from stretch_mujoco import StretchMujocoSimulator
from stretch_mujoco.enums.stretch_sensors import StretchSensors
from stretch_mujoco.enums.stretch_cameras import StretchCameras
from stretch_mujoco.enums.actuators import Actuators

### Camera & Streaming Set Up


In [35]:
# keep track of which windows we've created
_created_windows = set()

def show_camera_feeds_sync(sim, print_fps=False, init_size=(640, 480)):
    """
    Pull camera data from the simulator and display it using OpenCV.
    Windows are resizable (drag corners).
    """
    camera_data = sim.pull_camera_data()

    if print_fps:
        s = sim.pull_status()
        print(f"Physics fps: {s.fps}. Camera FPS: {camera_data.fps}. {s.sim_to_real_time_ratio_msg}")

    for cam_enum, pixels in camera_data.get_all(use_depth_color_map=True).items():
        if pixels is None:
            continue

        name = cam_enum.name  # window title

        # create a resizable window once per camera
        if name not in _created_windows:
            cv2.namedWindow(name, cv2.WINDOW_NORMAL)       # <-- resizable
            cv2.resizeWindow(name, *init_size)             # initial size
            # optional: keep aspect ratio when resizing
            try:
                cv2.setWindowProperty(name, cv2.WND_PROP_ASPECT_RATIO, cv2.WINDOW_KEEPRATIO)
            except Exception:
                pass
            _created_windows.add(name)

        cv2.imshow(name, pixels)

    # process window events (needed for resizing to take effect)
    cv2.waitKey(1)
    
_stream_evt = None
_stream_thread = None

def start_stream_thread(sim, print_fps=False, target_hz=20):
    """Run camera streaming in the background using your show_camera_feeds_sync()."""
    global _stream_evt, _stream_thread
    if _stream_thread and _stream_thread.is_alive():
        print("Stream already running.")
        return

    _stream_evt = threading.Event()

    def _worker():
        try:
            dt = 1.0 / max(1, target_hz)
            while not _stream_evt.is_set() and sim.is_running():
                show_camera_feeds_sync(sim, print_fps)
                # sim.step()                       # advance physics
                time.sleep(dt)                   # throttle display FPS a bit
        except Exception as e:
            print("stream thread ended:", type(e).__name__, e)
        finally:
            # Close any OpenCV windows cleanly
            try:
                cv2.destroyAllWindows()
            except: 
                pass

    _stream_thread = threading.Thread(target=_worker, daemon=True)
    _stream_thread.start()
    print("Stream thread started.")

def stop_stream_thread():
    """Stop the background stream cleanly (call before sim.stop())."""
    global _stream_evt, _stream_thread
    if _stream_evt:
        _stream_evt.set()
    if _stream_thread:
        _stream_thread.join(timeout=2.0)
    try:
        cv2.waitKey(1)
        cv2.destroyAllWindows()
    except:
        pass
    _stream_evt = None
    _stream_thread = None
    print("Stream thread stopped.")


### Testing for Robocasa Set up


In [36]:
# Core libs
import mujoco
import numpy as np

# RoboCasa generator
try:
    from stretch_mujoco.robocasa_gen import model_generation_wizard
    print("Found model_generation_wizard()")
except Exception as e:
    print("Could not import model_generation_wizard:", e)

# robosuite / robocasa sanity
import robosuite
print("robosuite version:", getattr(robosuite, "__version__", "unknown"))

import robocasa
print("robocasa version:", getattr(robocasa, "__version__", "unknown"))

# Show robosuite macro backend if present
try:
    from robosuite import macros as RS_MACROS
    print("robosuite MUJOCO_GL:", getattr(RS_MACROS, "MUJOCO_GL", "not set"))
except Exception as e:
    print("robosuite macros import issue:", e)


Found model_generation_wizard()
robosuite version: 1.5.1
robocasa version: 0.2.0
robosuite MUJOCO_GL: not set


### Set Up Mujoco Env


In [37]:
import math

# your measured base pose
x = 0.6713510702553015
y = -1.3006141423127842
theta = 3.1147300993742104

# wrap theta to [-pi, pi] (optional but nice)
theta = math.atan2(math.sin(theta), math.cos(theta))

# Use the z from the default fixture pose you saw printed once (replace this!)
z0 = 0.0  # <-- replace with the third value from the printed "Adding stretch..." pos

w = math.cos(theta / 2.0)
z = math.sin(theta / 2.0)

robot_spawn_pose = {
    "pos": f"{x} {y} {z0}",
    "quat": f"{w} 0 0 {z}",
}

In [38]:
mj_model = xml = objects_info = None

try:
    # Non interactive example. Adjust task/layout/style later if you want.
    # These names are common defaults; if they ever change, the except block will let you pick via wizard.
    mj_model, xml, objects_info = model_generation_wizard(
        task="PnPCounterToCab",
        layout=0,
        style=0,
        robot_spawn_pose=robot_spawn_pose,
    )
    print("Generated RoboCasa model non-interactively.")
except TypeError:
    # Some versions use only the interactive wizard
    print("Non-interactive args not supported. Opening interactive wizard...")
    mj_model, xml, objects_info = model_generation_wizard()
except FileNotFoundError as e:
    print("Asset missing:", e)
    print("Re-run the RoboCasa asset downloader script and try again.")
    raise
except Exception as e:
    print("RoboCasa scene generation failed:", e)
    raise

print("Model OK:", isinstance(mj_model, mujoco.MjModel))
if xml:
    print("XML length:", len(xml))
if objects_info is not None:
    # objects_info is usually a dict from the generator
    print("Objects info keys:", list(objects_info)[:5])


[robosuite INFO] Loading controller configuration from: /home/orrijoa/projects/stretch_mujoco_fork/third_party/robosuite/robosuite/controllers/config/robots/default_pandaomron.json (composite_controller_factory.py:121)


Initializing environment...
Initial observation keys: odict_keys(['robot0_joint_pos_cos', 'robot0_joint_pos_sin', 'robot0_joint_vel', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_eef_quat_site', 'robot0_gripper_qpos', 'robot0_gripper_qvel', 'robot0_base_pos', 'robot0_base_quat', 'robot0_base_to_eef_pos', 'robot0_base_to_eef_quat', 'robot0_base_to_eef_quat_site', 'apple0_pos', 'apple0_quat', 'apple0_to_robot0_eef_pos', 'apple0_to_robot0_eef_quat', 'robot0_proprio-state', 'object-state'])
env.object_cfgs after override:
0: name=apple0      cat=apple       model=/home/orrijoa/projects/stretch_mujoco_fork/third_party/robocasa/robocasa/models/assets/objects/objaverse/apple/apple_0/model.xml
Showing configuration:
    Layout: One wall
    Style: Industrial

Spawning environment...


Making Object Placements for task [PnPCounterToCab]...

Placing [Object 0] (category: apple, body_name: apple0_main) at pos: [ 2.12 -0.43  0.96] quat: [0.99 0.   0.   0.11]

Making Robot Placement...

Adding stre

In [39]:
# Pretty-print objects and their initial placements
for body_name, info in objects_info.items():
    print(f"{body_name:30s}  cat={info['cat']:12s}  pos={info['pos']}  quat={info['quat']}")

apple0_main                     cat=apple         pos=[ 0.7   -0.6    0.963]  quat=[0.994 0.    0.    0.11 ]


### Actual Testing with Tele Ops


In [40]:
# sim = StretchMujocoSimulator(cameras_to_use=StretchCameras.all())

# cameras_to_use = [StretchCameras.cam_nav_rgb]
# cameras_to_use = StretchCameras.all()
cameras_to_use = []

sim = StretchMujocoSimulator(model=mj_model, cameras_to_use=cameras_to_use)
sim.start(show_viewer_ui=False, headless=False)

Starting Stretch Mujoco Simulator...
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Using the Mujoco Passive Viewer. Note: UI thread and camera rendering is capped to 30.0Hz to increase performance. You can set this rate using the `camera_rate` arugment.
Still waiting to connect to the Mujoco Simulatior.
The Mujoco Simulatior is connected.


In [41]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO

from stretch_reach_env import StretchReachEnv

SEED = 0

def make_env():
    env = StretchReachEnv(
        sim,
        obj_name="apple0_main",
        dt=0.05,
        max_steps=150,          # Fetch-like
        success_thresh=0.06,
    )
    # Record is_success so SB3 logs it via Monitor
    env = Monitor(env, info_keywords=("is_success", "distance"))
    env.reset(seed=SEED)
    return env

def run_eval(model, env, n_steps=500, deterministic=True):
    reset_out = env.reset()
    obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    dists = []
    episode_successes = 0
    episode_count = 0

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)

        dists.append(info.get("distance", np.nan))
        if terminated or truncated:
            episode_count += 1
            episode_successes += int(info.get("is_success", 0.0) == 1.0)

            reset_out = env.reset()
            obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out
        
    mean_dist = float(np.nanmean(dists))
    success_rate = (episode_successes / max(1, episode_count))
    return mean_dist, episode_successes, episode_count, success_rate

def run_eval_with_stuck(model, env, n_steps=500, deterministic=True):
    reset_out = env.reset()
    obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    dists = []
    episode_successes = 0
    episode_count = 0
    stuck_steps = 0
    total_steps = 0

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)

        total_steps += 1
        dists.append(info.get("distance", np.nan))
        stuck_steps += int(bool(info.get("is_stuck", False)))

        if terminated or truncated:
            episode_count += 1
            episode_successes += int(info.get("is_success", 0.0) == 1.0)

            reset_out = env.reset()
            obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    mean_dist = float(np.nanmean(dists))
    success_rate = (episode_successes / max(1, episode_count))
    stuck_rate = stuck_steps / max(1, total_steps)
    return mean_dist, episode_successes, episode_count, success_rate, stuck_rate

def run_eval_with_stuck_debug(model, env, n_steps=500, deterministic=True, near_dist=0.25, arm_attempt_thresh=0.20):
    reset_out = env.reset()
    obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    dists = []
    episode_successes = 0
    episode_count = 0

    total_steps = 0
    stuck_steps = 0

    attempt_steps = 0
    stuck_steps_given_attempt = 0

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)

        total_steps += 1
        d = info.get("distance", np.nan)
        dists.append(d)

        is_stuck = bool(info.get("is_stuck", False))
        stuck_steps += int(is_stuck)

        # "Attempt" guard: either close-ish OR clearly extending the arm.
        arm_pos = float(info.get("arm_pos", 0.0))  # only works if env puts this in info
        is_attempt = (np.isfinite(d) and d < near_dist) or (arm_pos > arm_attempt_thresh)

        if is_attempt:
            attempt_steps += 1
            stuck_steps_given_attempt += int(is_stuck)

        if terminated or truncated:
            episode_count += 1
            episode_successes += int(info.get("is_success", 0.0) == 1.0)
            reset_out = env.reset()
            obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    mean_dist = float(np.nanmean(dists))
    success_rate = (episode_successes / max(1, episode_count))
    stuck_rate = stuck_steps / max(1, total_steps)
    attempt_rate = attempt_steps / max(1, total_steps)
    stuck_given_attempt = stuck_steps_given_attempt / max(1, attempt_steps)

    return mean_dist, episode_successes, episode_count, success_rate, stuck_rate, attempt_rate, stuck_given_attempt


### CHECK IF THE ROBOT HAVE ENOUGH TIME TO MOVE

In [42]:
# def episode_diagnostics(env, policy=None, deterministic=True, max_steps=None):
#     """
#     Logs: sim_time, lift/arm/grip, ee, dist over a single episode.
#     If policy is None, uses random actions.
#     """
#     obs, info = env.reset(seed=SEED)

#     rows = []
#     for t in range(int(max_steps or getattr(env, "max_steps", 100))):
#         st = env.debug_state()  # uses sim.pull_status() + ee/obj/dist
#         sim_time = float(env.sim.pull_status().time)  # StatusStretchJoints.time
#         rows.append({
#             "t": t,
#             "sim_time": sim_time,
#             "lift": st["lift"],
#             "arm": st["arm"],
#             "grip": st["gripper"],
#             "ee_x": st["ee"][0],
#             "ee_y": st["ee"][1],
#             "ee_z": st["ee"][2],
#             "dist": st["dist"],
#         })

#         if policy is None:
#             action = env.action_space.sample()
#         else:
#             action, _ = policy.predict(obs, deterministic=deterministic)

#         obs, reward, terminated, truncated, info = env.step(action)
#         if terminated or truncated:
#             break

#     # Convert to arrays
#     sim_time = np.array([r["sim_time"] for r in rows], dtype=np.float64)
#     dist = np.array([r["dist"] for r in rows], dtype=np.float64)
#     lift = np.array([r["lift"] for r in rows], dtype=np.float64)
#     arm  = np.array([r["arm"]  for r in rows], dtype=np.float64)
#     grip = np.array([r["grip"] for r in rows], dtype=np.float64)

#     # Step-to-step deltas
#     d_sim_time = np.diff(sim_time)
#     d_dist = np.diff(dist)

#     def frac_small(x, eps):
#         return float(np.mean(np.abs(x) < eps)) if x.size else 1.0

#     print("=== Episode diagnostics ===")
#     print(f"steps: {len(rows)}")
#     print(f"sim_time start/end: {sim_time[0]:.6f} -> {sim_time[-1]:.6f} (Δ={sim_time[-1]-sim_time[0]:.6f})")
#     if d_sim_time.size:
#         print(f"Δsim_time per step: mean={d_sim_time.mean():.6e}, min={d_sim_time.min():.6e}, max={d_sim_time.max():.6e}")
#         print(f"fraction steps with ~no sim_time advance (<1e-6): {frac_small(d_sim_time, 1e-6):.2%}")
#     if d_dist.size:
#         print(f"Δdist per step: mean={d_dist.mean():.6e}, min={d_dist.min():.6e}, max={d_dist.max():.6e}")
#         print(f"fraction steps with ~no dist change (<1e-5): {frac_small(d_dist, 1e-5):.2%}")

#     # Plots
#     fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
#     axs[0].plot(sim_time, dist, label="distance ||ee-obj||")
#     axs[0].set_ylabel("distance (m)")
#     axs[0].grid(True)
#     axs[0].legend()

#     axs[1].plot(sim_time, lift, label="lift")
#     axs[1].plot(sim_time, arm, label="arm")
#     axs[1].plot(sim_time, grip, label="gripper")
#     axs[1].set_xlabel("sim_time (s)")
#     axs[1].set_ylabel("joint pos")
#     axs[1].grid(True)
#     axs[1].legend()
#     plt.tight_layout()
#     plt.show()

#     return rows

In [43]:
# eval_env = StretchReachEnv(sim, obj_name="apple0_main", dt=0.05, max_steps=100, success_thresh=0.06)

In [44]:
# _ = episode_diagnostics(eval_env, policy=None, max_steps=100)

### MEASURE SIM V.S. REALITY RUNNING TIME

In [64]:
import time

s0 = sim.pull_status()
t_sim0 = float(s0.time)
t_wall0 = time.time()

target = t_sim0 + 5.0
while float(sim.pull_status().time) < target:
    time.sleep(0.1)  # was 0.01

s1 = sim.pull_status()
t_sim1 = float(s1.time)
t_wall1 = time.time()

print("sim advanced:", t_sim1 - t_sim0, "sec")
print("wall elapsed:", t_wall1 - t_wall0, "sec")
print("sim/wall:", (t_sim1 - t_sim0) / (t_wall1 - t_wall0))
print("status:", s1.sim_to_real_time_ratio_msg)

sim advanced: 5.014000000023941 sec
wall elapsed: 16.593217372894287 sec
sim/wall: 0.30217165769277027
status: Sim is running 0.305x as fast as realtime


### TRAINING THE MODEL

In [ ]:
# vec_env = DummyVecEnv([make_env])

# model = PPO(
#     policy="MultiInputPolicy",
#     env=vec_env,
#     seed=SEED,
#     verbose=1,
#     n_steps=256,          # small, safe default
#     batch_size=64,
#     learning_rate=3e-4,
#     gamma=0.98,
#  )

# model.learn(total_timesteps=1000)
# # Save the trained model
# model.save("ppo_stretch_reach_only_1000")
# print("Model saved to ppo_stretch_reach_only_1000.zip")

In [ ]:
# NOTE: total_timesteps is the number of env.step() transitions collected.
# Removing time.sleep() makes wall-clock faster but does NOT change what a "timestep" means here.
TOTAL_TIMESTEPS = 20_000  # quick sanity check; try 20_000+ once things look stable
vec_env = DummyVecEnv([make_env])

model = PPO(
    policy="MultiInputPolicy",
    env=vec_env,
    seed=SEED,
    verbose=1,
    n_steps=256,          # small, safe default
    batch_size=64,
    learning_rate=3e-4,
    gamma=0.98,
 )

model.learn(total_timesteps=TOTAL_TIMESTEPS)
# Save the trained model
model.save("ppo_stretch_reach_increased_max_steps_20000")
print("Model saved to ppo_stretch_reach.zip")

In [ ]:
# # Close vec env before making a new env that uses the same simulator
# try:
#     vec_env.close()
# except Exception:
#     pass

### LOAD PPO MODEL

In [ ]:
model = PPO.load("ppo_stretch_reach_increased_max_steps_20000")   # loads weights + optimizer state
# model = PPO.load("ppo_stretch_reach_only_1000") 

### SIMPLE MODEL EVALUATION

In [ ]:
eval_env = StretchReachEnv(sim, obj_name="apple0_main", dt=0.05, max_steps=150, success_thresh=0.06)
eval_env = Monitor(eval_env, info_keywords=("is_success", "distance"))

# Quick test: run a few predictions
obs, info = eval_env.reset(seed=SEED)

mean_dist, ep_succ, ep_cnt, succ_rate = run_eval(model, eval_env, n_steps=500, deterministic=True)
print(f"mean_dist={mean_dist:.4f} episodes={ep_cnt} episode_successes={ep_succ} success_rate={succ_rate:.2%}")

### TRAINING + EVALUATION TO TEST IMPROVEMENT GETTING STUCK AT THE TABLE LIP

In [16]:
vec_env = DummyVecEnv([make_env])

model = PPO(
    policy="MultiInputPolicy",
    env=vec_env,
    seed=SEED,
    verbose=0,
    n_steps=256,          # small, safe default
    batch_size=64,
    learning_rate=3e-4,
    gamma=0.98,
 )

eval_env = StretchReachEnv(sim, obj_name="apple0_main", dt=0.05, max_steps=150, success_thresh=0.06)
eval_env = Monitor(eval_env, info_keywords=("is_success", "distance"))

TOTAL = 10_000
CHUNK = 2_000

for t in range(0, TOTAL, CHUNK):
    model.learn(total_timesteps=CHUNK, reset_num_timesteps=False)

    mean_dist, ep_succ, ep_cnt, succ_rate, stuck_rate = run_eval_with_stuck(
        model, eval_env, n_steps=800, deterministic=True
    )
    print(
        f"t={t+CHUNK:6d}  mean_dist={mean_dist:.4f}  succ={succ_rate:.1%}  "
        f"episodes={ep_cnt}  stuck_rate={stuck_rate:.1%}"
    )

    # IMPORTANT: eval reset/home changed sim state; re-sync the training env
    vec_env.reset()

model.save(f"ppo_stretch_reach_improved_getting_stuck_{TOTAL}")

t=  2000  mean_dist=0.2794  succ=0.0%  episodes=5  stuck_rate=16.9%
t=  4000  mean_dist=0.2516  succ=0.0%  episodes=5  stuck_rate=0.0%
t=  6000  mean_dist=0.2039  succ=0.0%  episodes=5  stuck_rate=0.0%
t=  8000  mean_dist=0.2035  succ=0.0%  episodes=5  stuck_rate=0.0%
t= 10000  mean_dist=0.2154  succ=0.0%  episodes=5  stuck_rate=0.0%


In [19]:
mean_dist, ep_succ, ep_cnt, succ_rate, stuck_rate, attempt_rate, stuck_given_attempt = run_eval_with_stuck_debug(
    model, eval_env, n_steps=800, deterministic=True
)
print(
    f"t={t+CHUNK:6d} mean_dist={mean_dist:.4f} succ={succ_rate:.1%} episodes={ep_cnt} "
    f"stuck={stuck_rate:.1%} attempt={attempt_rate:.1%} stuck|attempt={stuck_given_attempt:.1%}"
)

t= 10000 mean_dist=0.2166 succ=0.0% episodes=5 stuck=0.0% attempt=66.5% stuck|attempt=0.0%


In [20]:
def probe_info_fields(model, env, n_steps=300, deterministic=True, near_dist=0.25, arm_attempt_thresh=0.20):
    reset_out = env.reset()
    obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    missing_is_stuck = 0
    missing_arm_pos = 0

    attempt_by_dist = 0
    attempt_by_arm = 0
    stuck_true = 0

    min_d = float("inf")
    frac_d_lt_0p15 = 0
    frac_d_lt_0p10 = 0
    frac_d_lt_0p06 = 0

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)

        d = info.get("distance", np.nan)
        if np.isfinite(d):
            min_d = min(min_d, float(d))
            frac_d_lt_0p15 += int(d < 0.15)
            frac_d_lt_0p10 += int(d < 0.10)
            frac_d_lt_0p06 += int(d < 0.06)

        if "is_stuck" not in info:
            missing_is_stuck += 1
        stuck_true += int(bool(info.get("is_stuck", False)))

        if "arm_pos" not in info:
            missing_arm_pos += 1
            arm_pos = 0.0
        else:
            arm_pos = float(info["arm_pos"])

        by_dist = (np.isfinite(d) and d < near_dist)
        by_arm = (arm_pos > arm_attempt_thresh)

        attempt_by_dist += int(by_dist)
        attempt_by_arm += int(by_arm)

        if terminated or truncated:
            reset_out = env.reset()
            obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    n = max(1, n_steps)
    print("missing is_stuck:", missing_is_stuck, "/", n)
    print("missing arm_pos :", missing_arm_pos, "/", n)
    print("attempt_by_dist :", attempt_by_dist / n)
    print("attempt_by_arm  :", attempt_by_arm / n)
    print("stuck_true_rate :", stuck_true / n)
    print("min distance    :", min_d)
    print("frac d<0.15     :", frac_d_lt_0p15 / n)
    print("frac d<0.10     :", frac_d_lt_0p10 / n)
    print("frac d<0.06     :", frac_d_lt_0p06 / n)

In [21]:
probe_info_fields(model, eval_env, n_steps=300)

missing is_stuck: 0 / 300
missing arm_pos : 0 / 300
attempt_by_dist : 0.7066666666666667
attempt_by_arm  : 0.37
stuck_true_rate : 0.0
min distance    : 0.12196960300207138
frac d<0.15     : 0.17
frac d<0.10     : 0.0
frac d<0.06     : 0.0


### ADDITIONAL MODEL TRAINING

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

vec_env = DummyVecEnv([make_env])

model = PPO.load("ppo_stretch_reach")   # loads weights + optimizer state
model.set_env(vec_env)

# Continue training without resetting the timestep counter
model.learn(total_timesteps=20_000, reset_num_timesteps=False)

model.save("ppo_stretch_reach")

### END SIMULATION

In [29]:
# 1) Stop the current headless sim (if running)
if sim.is_running():
    sim.stop()

Stopping Stretch Mujoco Simulator... simulated runtime= 1231.1s
Sending signal to stop the Mujoco process...
Physics Loop has terminated.
Stopping thread 1/1 on the Mujoco Process.
Mujoco viewer has terminated.
The Mujoco process has ended.
Stopping thread 1/6.
IOPub is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 2/6.
Heartbeat is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 3/6.
Thread-1 (_watch_pipe_fd) is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 4/6.
Thread-2 (_watch_pipe_fd) is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 5/6.
Control is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 6/6.
IPythonHistorySavingThread is not terminating. Make sure to check 'sim.is_running()' in threading loops.
The Stretch Mujoco Simulator has ended. Good-bye!


### Manual ENV Testing


In [45]:
%load_ext autoreload
%autoreload 2
from stretch_reach_env import StretchReachEnv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
env = StretchReachEnv(
    sim,
    obj_name="apple0_main",
    dt=0.05,
    max_steps=50,          # you can use 50 to match Fetch
    success_thresh=0.06,
)

In [47]:
obs, info = env.reset()

print("reset info:", info)
print("obs keys:", obs.keys())
print("observation shape:", obs["observation"].shape, obs["observation"].dtype)
print("achieved_goal shape:", obs["achieved_goal"].shape, obs["achieved_goal"].dtype)
print("desired_goal shape:", obs["desired_goal"].shape, obs["desired_goal"].dtype)

reset info: {}
obs keys: dict_keys(['observation', 'achieved_goal', 'desired_goal'])
observation shape: (9,) float32
achieved_goal shape: (3,) float32
desired_goal shape: (3,) float32


In [48]:
print(env._get_ee_pos())
print(env._get_obj_pos())

[ 0.702 -0.788  0.699]
[ 0.703 -0.599  0.952]


In [61]:
for i in range(1):
    obs, r, term, trunc, info = env.step([0.5, 0, 0])
    s = sim.pull_status()
    print(f"lift i={i} lift={s.lift.pos:.6f} ee_z={obs['achieved_goal'][2]:.6f} dist={info['distance']:.6f}")

lift i=0 lift=0.829351 ee_z=0.937654 dist=0.189950


In [63]:
for i in range(20):
    obs, r, term, trunc, info = env.step([0, 1, 0])
    s = sim.pull_status()
    print(f"arm  i={i} t={float(s.time):.3f} arm={s.arm.pos:.6f} lift={s.lift.pos:.4f} ee_x={obs['achieved_goal'][0]:.6f} dist={info['distance']:.6f}")

arm  i=0 t=73.600 arm=0.293207 lift=0.8398 ee_x=0.704864 dist=0.034172
arm  i=1 t=73.652 arm=0.295525 lift=0.8399 ee_x=0.704800 dist=0.034421
arm  i=2 t=73.704 arm=0.299268 lift=0.8400 ee_x=0.704702 dist=0.033493
arm  i=3 t=73.756 arm=0.304715 lift=0.8400 ee_x=0.704797 dist=0.033502
arm  i=4 t=73.808 arm=0.311863 lift=0.8400 ee_x=0.704923 dist=0.033521
arm  i=5 t=73.860 arm=0.320339 lift=0.8401 ee_x=0.705069 dist=0.033519
arm  i=6 t=73.912 arm=0.329793 lift=0.8401 ee_x=0.705229 dist=0.033549
arm  i=7 t=73.964 arm=0.339847 lift=0.8401 ee_x=0.705432 dist=0.033685
arm  i=8 t=74.016 arm=0.350281 lift=0.8401 ee_x=0.705507 dist=0.035374
arm  i=9 t=74.068 arm=0.360967 lift=0.8402 ee_x=0.705626 dist=0.033435
arm  i=10 t=74.120 arm=0.371494 lift=0.8402 ee_x=0.705807 dist=0.033608
arm  i=11 t=74.172 arm=0.381846 lift=0.8402 ee_x=0.705987 dist=0.033667
arm  i=12 t=74.224 arm=0.392086 lift=0.8402 ee_x=0.706163 dist=0.033750
arm  i=13 t=74.276 arm=0.397188 lift=0.8409 ee_x=0.706231 dist=0.035843
ar

In [27]:
for i in range(100):
    obs, r, term, trunc, info = env.step([0, 0, 1])
    s = sim.pull_status()
    print(f"grip i={i} grip={s.gripper.pos:.6f}")

grip i=0 grip=-0.124478
grip i=1 grip=-0.119800
grip i=2 grip=-0.115577
grip i=3 grip=-0.111463
grip i=4 grip=-0.107545
grip i=5 grip=-0.103789
grip i=6 grip=-0.100196
grip i=7 grip=-0.096756
grip i=8 grip=-0.093464
grip i=9 grip=-0.090312
grip i=10 grip=-0.087296
grip i=11 grip=-0.084409
grip i=12 grip=-0.081646
grip i=13 grip=-0.079001
grip i=14 grip=-0.076470
grip i=15 grip=-0.074046
grip i=16 grip=-0.071727
grip i=17 grip=-0.069507
grip i=18 grip=-0.067382
grip i=19 grip=-0.065348
grip i=20 grip=-0.063402
grip i=21 grip=-0.061538
grip i=22 grip=-0.059755
grip i=23 grip=-0.058048
grip i=24 grip=-0.056414
grip i=25 grip=-0.054850
grip i=26 grip=-0.053353
grip i=27 grip=-0.051920
grip i=28 grip=-0.050548
grip i=29 grip=-0.049234
grip i=30 grip=-0.047975
grip i=31 grip=-0.046768
grip i=32 grip=-0.045611
grip i=33 grip=-0.044500
grip i=34 grip=-0.043433
grip i=35 grip=-0.042408
grip i=36 grip=-0.041422
grip i=37 grip=-0.040472
grip i=38 grip=-0.039556
grip i=39 grip=-0.038671
grip i=40 

In [ ]:
for i in range(20):
    a = env.action_space.sample()  # random action in [-1,1]
    
    obs, reward, terminated, truncated, info = env.step(a)
    
    st = env.debug_state()  # right after step
    print(
        f"i={i:02d} a={a} lift={st['lift']:.3f} arm={st['arm']:.3f} grip={st['gripper']:.3f} "
        f"ee={st['ee'].round(3)} obj={st['obj'].round(3)} dist={st['dist']:.4f}"
    )

    if terminated or truncated:
        print("DONE. is_success:", info.get("is_success"), "has terminal_obs:", "terminal_observation" in info)
        break


### GET OBJ STATE (APPLE)


In [ ]:
sim.register_tracked_objects(list(objects_info.keys()))

In [ ]:
obj_name = "apple0_main"

In [ ]:
obj_state = sim.pull_objects_state([obj_name])
print(obj_state)

In [ ]:
limits = {
    Actuators.lift: (0.0, 1.1),
    Actuators.arm:  (0.0, 0.52),
    Actuators.gripper: (-0.25, 0.53),
}

# how long we wait between actions.
dt = 0.05
# action scales (start small, tune later)
scales = np.array([0.03, 0.03, 0.02], dtype=np.float32)  # [lift, arm, gripper]

In [ ]:
def get_ee_pos(sim):
    T = sim.get_ee_pose()
    return T[:3, 3].astype(float)

def get_obj_pos(sim, obj_name):
    return sim.pull_objects_state()[obj_name]["pos"].astype(float)

def distance_to_object(sim, obj_name):
    ee_pos = get_ee_pos(sim)
    obj_pos = get_obj_pos(sim, obj_name)
    d = float(np.linalg.norm(ee_pos - obj_pos))
    return d, ee_pos, obj_pos

def manual_step(sim, a, obj_name):
    """
    a: array-like shape (3,), values in [-1, 1]
    returns: d, ee_pos, obj_pos
    """
    a = np.asarray(a, dtype=np.float32)
    a = np.clip(a, -1.0, 1.0) # a is just the policy saying “move up/down a bit” (a direction and strength).
    delta = a * scales # scales is shape (3,)

    s = sim.pull_status()
    current_lift = float(s.lift.pos)
    current_arm  = float(s.arm.pos)
    current_grip = float(s.gripper.pos)
    
    lift_low, lift_high = limits[Actuators.lift]
    arm_low,  arm_high  = limits[Actuators.arm]
    grip_low, grip_high = limits[Actuators.gripper]
    
    target_lift = np.clip(current_lift + float(delta[0]), lift_low, lift_high)
    target_arm  = np.clip(current_arm  + float(delta[1]), arm_low,  arm_high)
    target_grip = np.clip(current_grip + float(delta[2]), grip_low, grip_high)

    sim.move_to(Actuators.lift,    target_lift)
    sim.move_to(Actuators.arm,     target_arm)
    sim.move_to(Actuators.gripper, target_grip)

    # fixed control tick
    time.sleep(dt)

    return distance_to_object(sim, obj_name)


In [ ]:
distance_to_object(sim, obj_name)

In [ ]:
# move only lift up
for i in range(5):
    d, ee, obj = manual_step(sim, a=[1, 0, 0], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} lift={s.lift.pos:.6f} ee_z={ee[2]:.6f}")

In [ ]:
# move only arm forward
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 1, 0], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} arm={s.arm.pos:.6f} ee_x={ee[0]:.6f} ee_z={ee[2]:.6f}")

In [ ]:
# move only gripper (try open)
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 0, 1], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} gripper={s.gripper.pos:.6f}")


In [ ]:
# move only gripper (try close)
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 0, -1], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} gripper={s.gripper.pos:.6f}")


In [ ]:
start_stream_thread(sim, print_fps=False, target_hz=20)

### ROBOT MANIPULATION


In [ ]:
sim.home()

In [ ]:
# from -2.02 to 0.49 (document) - tested use this value
# from -1.53 to 0.79 (current set up) - correct

sim.move_to(Actuators.head_tilt, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.head_tilt, 0.2)
time.sleep(0.5)

In [ ]:
# from -4.04 to 1.73
sim.move_to(Actuators.head_pan, 1.73)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.head_pan, -0.2)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_translate, 0.05)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_translate, -0.05)
time.sleep(0.5)

In [ ]:
# 90 degree ~= 1.57
sim.move_by(Actuators.base_rotate, 1.5)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_rotate, -0.1)
time.sleep(0.5)

In [16]:
# from 0 to 1.1 - tested
sim.move_to(Actuators.lift, 0.7)
time.sleep(0.5)

In [86]:
sim.move_by(Actuators.lift, -0.05)
time.sleep(0.5)

In [ ]:
# from 0 to 0.13 (current)
# from 0 to 0.52 (document) - correct - tested
sim.move_to(Actuators.arm, 0.0)
time.sleep(0.5)

In [101]:
sim.move_by(Actuators.arm, 0.05)
time.sleep(0.5)

In [ ]:
# (-1.39, 4.42) - tested
sim.move_to(Actuators.wrist_yaw, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_yaw, -0.2)
time.sleep(0.5)

In [ ]:
# (-1.57, 0.56) current - correct - tested
# (-1.57, 0.57) document
sim.move_to(Actuators.wrist_pitch, 0.0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_pitch, -0.2)
time.sleep(0.5)

In [ ]:
# (-3.14, 3.14)
sim.move_to(Actuators.wrist_roll, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_roll, -0.2)
time.sleep(0.5)

In [ ]:
# (-0.02, 0.04)
# (-0.3,0.55) - tested
sim.move_to(Actuators.gripper, -0.3)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.gripper, 0.05)
time.sleep(0.5)

In [ ]:
# # Try to force it "very closed"
# sim.move_to(Actuators.gripper, -1.0)
# time.sleep(1)
# print("closed?", sim.pull_status().gripper.pos)

# # Try to force it "very open"
# sim.move_to(Actuators.gripper,  1.0)
# time.sleep(1)
# print("open?", sim.pull_status().gripper.pos)


### Report current status and limits


In [ ]:
pprint(sim.pull_status())

In [78]:
pprint(sim.get_ee_pose())

array([[ 0.019, -1.   ,  0.   ,  0.703],
       [ 1.   ,  0.019,  0.005, -0.789],
       [-0.005, -0.   ,  1.   ,  0.994],
       [ 0.   ,  0.   ,  0.   ,  1.   ]])


In [22]:
pprint(sim.register_tracked_objects(["apple0_main"]))

None


In [43]:
pprint(sim.pull_all_objects_state())

{'apple0_main': {'pos': array([ 0.703, -0.58 ,  0.952]),
                 'quat': array([ 0.945, -0.026,  0.041,  0.323])}}


In [ ]:
pprint(sim.pull_camera_data())



In [ ]:
# radar related
pprint(sim.pull_sensor_data())

In [ ]:
print(sim.pull_joint_limits())

### End the simulation


In [ ]:
# # Try to undo any previous manual wrapping if you still have the originals
# try:
#     sys.stdout = _sys_stdout_orig
#     sys.stderr = _sys_stderr_orig
# except NameError:
#     pass

# 1) stop background streaming first
stop_stream_thread()

# 2) give the sim process a short moment to finish its own threads
time.sleep(0.1)

# 3) stop the simulator
if sim.is_running():
    sim.stop()

# 4) as a final sweep, make sure no OpenCV windows remain
try:
    cv2.waitKey(1)
    cv2.destroyAllWindows()
except:
    pass
